# Lab 04 — Build your quantizer, break it, fix it

**Goal:** implement the proposal's core algorithm (per-block symmetric quantization + optional stochastic rounding + dequantize), then run four experiments:
- **A.** error vs bit-depth on clean data
- **B.** one outlier destroys a shared scale; per-block scales quarantine it
- **C.** round-to-nearest is biased; stochastic rounding is unbiased
- **D.** the **forward bit-depth cliff** on a real model — your empirical ε_fwd

In [ ]:
!pip -q install transformers matplotlib
import torch, matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()

## The quantizer (12 lines that are your dissertation's heart)
Quantize→dequantize in one function (a *fake-quant*: it injects exactly the noise the wire would, which is all we need for studying quality; Lab 08 handles timing, Phase 3 handles real bit-packing).

In [ ]:
def quantize(x, bits, block=64, stochastic=False):
    if bits >= 16: return x
    shape = x.shape
    flat = x.float().flatten()
    pad = (-flat.numel()) % block
    if pad: flat = torch.cat([flat, flat.new_zeros(pad)])
    fb = flat.view(-1, block)
    qmax = 2**(bits-1) - 1
    s = fb.abs().amax(1, keepdim=True) / qmax + 1e-12      # per-block scale
    y = fb / s
    q = torch.floor(y + torch.rand_like(y)) if stochastic else torch.round(y)
    q = q.clamp(-qmax, qmax)
    return (q * s).flatten()[:x.numel()].view(shape).to(x.dtype)

rmse = lambda x, xq: (x.float() - xq.float()).pow(2).mean().sqrt().item()

## A. Error vs bit-depth (clean Gaussian)

In [ ]:
torch.manual_seed(0)
x = torch.randn(1<<16, device=device)
for b in [8, 6, 4, 3, 2]:
    print(f"{b}-bit  rmse={rmse(x, quantize(x, b)):.4f}")

Each bit removed roughly doubles the error (grid spacing doubles). Now the villain:

## B. One outlier vs the grid

In [ ]:
x = torch.randn(4096, device=device)
print("clean,   shared scale :", rmse(x, quantize(x, 4, block=4096)))
x[0] = 50.0
print("outlier, shared scale :", rmse(x, quantize(x, 4, block=4096)), "  <- grid destroyed for everyone")
print("outlier, block=64     :", rmse(x, quantize(x, 4, block=64)),   "  <- quarantined")

One value made the shared 4-bit grid ~14x coarser for all 4096. Per-block scales (TAH-Quant's 'tiles') confine the damage to one block of 64. This is why your kernels compute per-block scales on the fly.

## C. Bias — why stochastic rounding exists
SGD tolerates unbiased noise (just variance) but not bias (systematic drift). Measure the **mean** error of both rounding modes.

In [ ]:
x = torch.rand(1<<14, device=device) * 0.1 + 1.0     # values sitting between grid points
rtn  = torch.stack([(quantize(x, 4) - x).mean() for _ in range(50)])
sto  = torch.stack([(quantize(x, 4, stochastic=True) - x).mean() for _ in range(50)])
print(f"round-to-nearest  mean error: {rtn.mean().item():+.5f}  (consistent sign = bias)")
print(f"stochastic        mean error: {sto.mean().item():+.5f}  (hovers near zero = unbiased)")

## D. The forward cliff — quantize a real boundary, measure quality
We use a hook that **replaces** block 5's output with its quantized version — simulating a 2-stage pipeline cut with d_fwd bits on the wire — and measure cross-entropy on held-out text.

In [ ]:
eval_text = ("The history of distributed computing begins with time-sharing systems, "
             "where many users shared one machine. Networks of workstations followed, "
             "and with them the first serious study of communication cost. ") * 8
eids = tok(eval_text, return_tensors="pt").to(device)

def ce():
    with torch.no_grad():
        return model(eids.input_ids, labels=eids.input_ids).loss.item()

CUT = 5
def make_qhook(bits):
    def fn(module, inp, out):
        return (quantize(out[0], bits),) + tuple(out[1:])
    return fn

results = {}
for b in [16, 8, 6, 4, 3, 2]:
    h = model.transformer.h[CUT].register_forward_hook(make_qhook(b))
    results[b] = ce()
    h.remove()
    print(f"d_fwd={b:2d} bits  cross-entropy={results[b]:.4f}")

In [ ]:
bits = sorted(results, reverse=True)
plt.figure(figsize=(6,3.5))
plt.plot(bits, [results[b] for b in bits], marker="o")
plt.gca().invert_xaxis()
plt.xlabel("forward bit-depth at the boundary"); plt.ylabel("cross-entropy (lower=better)")
plt.title("The forward cliff — quality vs d_fwd"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

Typical shape: flat from 16 down to ~6–4 bits, then a cliff. The flat region is free bandwidth; the cliff is your ε_fwd floor. **In Lab 05 you'll produce the matching backward curve and see the cliff arrive earlier — the direction-asymmetry claim, demonstrated.**

## Exercises
1. Move the cut (CUT=1 vs CUT=10). Which boundary is more sensitive? Relate to Lab 03's outlier ratios.
2. Add `stochastic=True` in the hook — does the cliff move? Why is stochastic rounding more important for the *backward* direction than forward?
3. Sweep `block` in [16, 64, 256, 4096] at 4 bits and plot CE vs block size — the granularity/overhead trade.
4. Implement zero-point (asymmetric-grid) quantization and test on `torch.relu(x)` — and note this literature meaning of 'asymmetric' vs the proposal's 'direction-asymmetric'.